## EDA on NYC Taxi Data

In [19]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

BRONZE_DIR = PROJECT_ROOT / "data" / "bronze"
parquet_files = sorted(BRONZE_DIR.glob("*.parquet"))

print(f"Project root: {PROJECT_ROOT}")
print(f"Bronze dir: {BRONZE_DIR}")
print(f"Found {len(parquet_files)} parquet file(s):")
for path in parquet_files:
    print(f" - {path.name}")

Project root: /home/kimkosal-y/learning-projects/nyc-taxi-mlops
Bronze dir: /home/kimkosal-y/learning-projects/nyc-taxi-mlops/data/bronze
Found 2 parquet file(s):
 - yellow_tripdata_2025-12.parquet
 - yellow_tripdata_2026-01.parquet


### Loading the data into Polars DataFrame

In [20]:
import polars as pl

all_data_df = pl.scan_parquet(str(BRONZE_DIR / "*.parquet")).collect()

print("All data:")
print(all_data_df.shape[0])
display(all_data_df.head())

print(f"There are {all_data_df.shape[0]} rows and {all_data_df.shape[1]} columns in total.")

All data:
8029895


VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
i32,datetime[μs],datetime[μs],i64,f64,i64,str,i32,i32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
1,2025-12-01 00:37:08,2025-12-01 00:51:10,1,2.4,1,"""N""",140,48,1,14.2,4.25,0.5,4.0,0.0,1.0,23.95,2.5,0.0,0.75
2,2025-12-01 00:03:54,2025-12-01 00:19:18,1,8.37,1,"""N""",138,262,1,33.1,6.0,0.5,7.77,6.94,1.0,59.56,2.5,1.75,0.0
2,2025-12-01 00:40:50,2025-12-01 01:06:54,1,15.26,1,"""N""",132,255,1,57.6,1.0,0.5,12.02,0.0,1.0,73.87,0.0,1.75,0.0
1,2025-12-01 00:21:30,2025-12-01 00:49:35,2,18.4,2,"""N""",132,79,1,70.0,5.0,0.5,10.0,0.0,1.0,86.5,2.5,1.75,0.75
2,2025-12-01 00:00:24,2025-12-01 00:03:34,1,0.52,1,"""N""",239,238,1,5.8,1.0,0.5,1.62,0.0,1.0,12.42,2.5,0.0,0.0


There are 8029895 rows and 20 columns in total.


In [21]:
# Dataset Description
display(all_data_df.describe())

statistic,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
str,f64,str,str,f64,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",8.029895e6,"""8029895""","""8029895""",5.746355e6,8.029895e6,5.746355e6,"""5746355""",8.029895e6,8.029895e6,8.029895e6,8.029895e6,8.029895e6,8.029895e6,8.029895e6,8.029895e6,8.029895e6,8.029895e6,5.746355e6,5.746355e6,8.029895e6
"""null_count""",0.0,"""0""","""0""",2.28354e6,0.0,2.28354e6,"""2283540""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.28354e6,2.28354e6,0.0
"""mean""",1.876579,"""2025-12-30 10:19:48.283488""","""2025-12-30 10:37:56.347498""",1.287381,6.715955,4.989623,null,161.62031,161.070808,0.860847,21.561968,1.049352,0.482477,2.726113,0.505629,0.948632,30.088949,2.16116,0.147876,0.523697
"""std""",0.714844,null,null,0.710824,687.049531,19.111267,null,66.870129,70.849095,0.716466,19.441307,1.734046,0.121812,4.022038,2.14226,0.2655,23.2544,0.939715,0.528372,0.355914
"""min""",1.0,"""2025-11-30 20:48:56""","""2025-11-30 20:57:48""",0.0,0.0,1.0,"""N""",1.0,1.0,0.0,-2555.2,-7.5,-0.5,-96.5,-106.93,-1.0,-2560.2,-2.5,-1.75,-0.75
"""25%""",2.0,"""2025-12-13 16:46:35""","""2025-12-13 17:08:31""",1.0,1.0,1.0,null,114.0,107.0,0.0,10.0,0.0,0.5,0.0,0.0,1.0,17.28,2.5,0.0,0.0
"""50%""",2.0,"""2025-12-29 11:24:49""","""2025-12-29 11:43:05""",1.0,1.81,1.0,null,161.0,162.0,1.0,16.3,0.0,0.5,2.0,0.0,1.0,23.8,2.5,0.0,0.75
"""75%""",2.0,"""2026-01-15 20:21:07""","""2026-01-15 20:36:01""",1.0,3.76,1.0,null,233.0,234.0,1.0,27.25,2.5,0.5,3.87,0.0,1.0,35.2,2.5,0.0,0.75
"""max""",7.0,"""2026-02-01 00:45:01""","""2026-02-01 23:35:31""",9.0,322576.17,99.0,"""Y""",265.0,265.0,4.0,2555.2,17.46,96.0,800.0,122.22,1.0,2560.2,2.5,26.75,0.75


### Adding Trip Duration Minutes Column

In [22]:
all_data_df = all_data_df.with_columns(
    (
        (pl.col("tpep_dropoff_datetime") - pl.col("tpep_pickup_datetime"))
        .dt.total_seconds() / 60
    ).alias("trip_duration_minutes")
)

display(
    all_data_df.select(
        [
            "tpep_pickup_datetime",
            "tpep_dropoff_datetime",
            "trip_duration_minutes",
            "trip_distance",
            "fare_amount",
            "total_amount",
        ]
    ).head()
)

tpep_pickup_datetime,tpep_dropoff_datetime,trip_duration_minutes,trip_distance,fare_amount,total_amount
datetime[μs],datetime[μs],f64,f64,f64,f64
2025-12-01 00:37:08,2025-12-01 00:51:10,14.033333,2.4,14.2,23.95
2025-12-01 00:03:54,2025-12-01 00:19:18,15.4,8.37,33.1,59.56
2025-12-01 00:40:50,2025-12-01 01:06:54,26.066667,15.26,57.6,73.87
2025-12-01 00:21:30,2025-12-01 00:49:35,28.083333,18.4,70.0,86.5
2025-12-01 00:00:24,2025-12-01 00:03:34,3.166667,0.52,5.8,12.42


In [23]:
print("Columns:")
for column in all_data_df.columns:
    print(f" - {column}")

display(all_data_df.schema)

Columns:
 - VendorID
 - tpep_pickup_datetime
 - tpep_dropoff_datetime
 - passenger_count
 - trip_distance
 - RatecodeID
 - store_and_fwd_flag
 - PULocationID
 - DOLocationID
 - payment_type
 - fare_amount
 - extra
 - mta_tax
 - tip_amount
 - tolls_amount
 - improvement_surcharge
 - total_amount
 - congestion_surcharge
 - Airport_fee
 - cbd_congestion_fee
 - trip_duration_minutes


Schema([('VendorID', Int32),
        ('tpep_pickup_datetime', Datetime(time_unit='us', time_zone=None)),
        ('tpep_dropoff_datetime', Datetime(time_unit='us', time_zone=None)),
        ('passenger_count', Int64),
        ('trip_distance', Float64),
        ('RatecodeID', Int64),
        ('store_and_fwd_flag', String),
        ('PULocationID', Int32),
        ('DOLocationID', Int32),
        ('payment_type', Int64),
        ('fare_amount', Float64),
        ('extra', Float64),
        ('mta_tax', Float64),
        ('tip_amount', Float64),
        ('tolls_amount', Float64),
        ('improvement_surcharge', Float64),
        ('total_amount', Float64),
        ('congestion_surcharge', Float64),
        ('Airport_fee', Float64),
        ('cbd_congestion_fee', Float64),
        ('trip_duration_minutes', Float64)])

In [24]:
pl.Config.set_tbl_rows(30)

polars.config.Config

In [25]:
all_data_df.select(pl.all().null_count()).transpose(
    include_header=True,
    header_name="column",
    column_names=['null_count']
).with_columns(
    (pl.col("null_count") / all_data_df.height * 100).round(2).alias("null_pct")
).sort("null_count", descending=True)

column,null_count,null_pct
str,u32,f64
"""passenger_count""",2283540,28.44
"""RatecodeID""",2283540,28.44
"""store_and_fwd_flag""",2283540,28.44
"""congestion_surcharge""",2283540,28.44
"""Airport_fee""",2283540,28.44
"""VendorID""",0,0.0
"""tpep_pickup_datetime""",0,0.0
"""tpep_dropoff_datetime""",0,0.0
"""trip_distance""",0,0.0


In [26]:

plot_df = (
    all_data_df.filter(
        (pl.col("trip_duration_minutes") > 0) 
        & (pl.col("trip_duration_minutes") <= 180) 
        & (pl.col("trip_distance") > 0)
        & (pl.col("trip_distance") <= 50)
    )
).with_columns(
    pl.col("tpep_pickup_datetime").dt.hour().alias("pickup_hour"),
    pl.col("tpep_pickup_datetime").dt.weekday().alias("pickup_weekday"),
    pl.col("tpep_pickup_datetime").dt.date().alias("pickup_date")
).select(
    "trip_duration_minutes",
    "trip_distance",
    "passenger_count",
    "pickup_hour",
    "pickup_weekday",
    "pickup_date",
    "PULocationID",
    "DOLocationID",
)

# display(plot_df.head())

In [27]:
duration_hist = (
    all_data_df
    .filter(
        (pl.col("trip_duration_minutes") > 0)
        & (pl.col("trip_duration_minutes") <= 180)
    )
    .with_columns(
        (pl.col("trip_duration_minutes") // 5 * 5).alias("duration_bin_start")
    )
    .group_by("duration_bin_start")
    .agg(
        pl.len().alias("trip_count")
    )
    .sort("duration_bin_start")
)

duration_hist.plot.bar(
    x="duration_bin_start",
    y="trip_count",
)

alt.Chart(...)

In [28]:
all_data_df.shape

# print rows and shapes and format with , like million and thousand
print(f"Total rows: {all_data_df.shape[0]:,}, Total columns: {all_data_df.shape[1]:,}")

Total rows: 8,029,895, Total columns: 21


In [29]:
import altair as alt

alt.Chart(duration_hist).mark_bar().encode(
    x=alt.X("duration_bin_start:Q", title="Trip duration bin start"),
    y=alt.Y("trip_count:Q", title="Trip count"),
    tooltip=["duration_bin_start", "trip_count"],
).properties(
    width=700,
    height=400,
    title="Trip Duration Distribution"
)

alt.Chart(...)

In [30]:
hour_labels = pl.DataFrame({
    "pickup_hour": list(range(24)),
    "pickup_hour_label": [
        "12 AM", "1 AM", "2 AM", "3 AM", "4 AM", "5 AM",
        "6 AM", "7 AM", "8 AM", "9 AM", "10 AM", "11 AM",
        "12 PM", "1 PM", "2 PM", "3 PM", "4 PM", "5 PM",
        "6 PM", "7 PM", "8 PM", "9 PM", "10 PM", "11 PM",
    ],
})


hourly_counts = (
    all_data_df
    .filter(
        (pl.col("trip_duration_minutes") > 0)
        & (pl.col("trip_duration_minutes") <= 180)
    )
    .with_columns(
        pl.col("tpep_pickup_datetime").dt.hour().alias("pickup_hour")
    )
    .group_by("pickup_hour")
    .agg(
        pl.len().alias("trip_count")
    )
    .sort("pickup_hour").join(hour_labels, on="pickup_hour", how="left")
)



alt.Chart(hourly_counts).mark_bar().encode(
    x=alt.X("pickup_hour_label:O", title="Pickup hour", sort=alt.EncodingSortField(field="pickup_hour", order="ascending")),
    y=alt.Y("trip_count:Q", title="Trip count"),
    tooltip=["pickup_hour_label", "trip_count"],
).properties(
    width=700,
    height=400,
    title="Trip Count by Pickup Hour"
)

alt.Chart(...)

In [31]:
# Reusable filtered dataset for downstream EDA
eda_df = all_data_df.filter(
    (pl.col("trip_duration_minutes") > 0)
    & (pl.col("trip_duration_minutes") <= 180)
    & (pl.col("trip_distance") > 0)
    & (pl.col("trip_distance") <= 50)
    & (pl.col("fare_amount") >= 0)
    & (pl.col("total_amount") >= 0)
).with_columns(
    pl.col("tpep_pickup_datetime").dt.hour().alias("pickup_hour"),
    pl.col("tpep_pickup_datetime").dt.weekday().alias("pickup_weekday"),
    pl.col("tpep_pickup_datetime").dt.date().alias("pickup_date"),
    pl.when(pl.col("tpep_pickup_datetime").dt.weekday().is_in([5, 6]))
    .then(True)
    .otherwise(False)
    .alias("is_weekend"),
)

print(f"EDA rows after basic filtering: {eda_df.height:,}")
display(eda_df.head())

EDA rows after basic filtering: 7,564,562


VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,trip_duration_minutes,pickup_hour,pickup_weekday,pickup_date,is_weekend
i32,datetime[μs],datetime[μs],i64,f64,i64,str,i32,i32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i8,i8,date,bool
1,2025-12-01 00:37:08,2025-12-01 00:51:10,1,2.4,1,"""N""",140,48,1,14.2,4.25,0.5,4.0,0.0,1.0,23.95,2.5,0.0,0.75,14.033333,0,1,2025-12-01,false
2,2025-12-01 00:03:54,2025-12-01 00:19:18,1,8.37,1,"""N""",138,262,1,33.1,6.0,0.5,7.77,6.94,1.0,59.56,2.5,1.75,0.0,15.4,0,1,2025-12-01,false
2,2025-12-01 00:40:50,2025-12-01 01:06:54,1,15.26,1,"""N""",132,255,1,57.6,1.0,0.5,12.02,0.0,1.0,73.87,0.0,1.75,0.0,26.066667,0,1,2025-12-01,false
1,2025-12-01 00:21:30,2025-12-01 00:49:35,2,18.4,2,"""N""",132,79,1,70.0,5.0,0.5,10.0,0.0,1.0,86.5,2.5,1.75,0.75,28.083333,0,1,2025-12-01,false
2,2025-12-01 00:00:24,2025-12-01 00:03:34,1,0.52,1,"""N""",239,238,1,5.8,1.0,0.5,1.62,0.0,1.0,12.42,2.5,0.0,0.0,3.166667,0,1,2025-12-01,false


## Additional EDA Checks and ML-Oriented Summaries
The notebook now needs a few high-value follow-up sections:
- data quality and invalid-value checks
- temporal trend summaries
- fare and distance relationship analysis
- feature candidates for future modeling

In [32]:
quality_checks = pl.DataFrame(
    {
        "metric": [
            "total_rows",
            "duplicate_rows",
            "null_pickup_datetime",
            "null_dropoff_datetime",
            "non_positive_trip_distance",
            "non_positive_trip_duration",
            "negative_fare_amount",
            "negative_total_amount",
            "zero_distance_positive_fare",
            "zero_duration_positive_distance",
        ],
        "value": [
            all_data_df.height,
            all_data_df.height - all_data_df.unique().height,
            all_data_df.filter(pl.col("tpep_pickup_datetime").is_null()).height,
            all_data_df.filter(pl.col("tpep_dropoff_datetime").is_null()).height,
            all_data_df.filter(pl.col("trip_distance") <= 0).height,
            all_data_df.filter(pl.col("trip_duration_minutes") <= 0).height,
            all_data_df.filter(pl.col("fare_amount") < 0).height,
            all_data_df.filter(pl.col("total_amount") < 0).height,
            all_data_df.filter((pl.col("trip_distance") == 0) & (pl.col("fare_amount") > 0)).height,
            all_data_df.filter((pl.col("trip_duration_minutes") == 0) & (pl.col("trip_distance") > 0)).height,
        ],
    }
).with_columns(
    (pl.col("value") / all_data_df.height * 100).round(4).alias("pct_of_rows")
)

quality_checks

metric,value,pct_of_rows
str,i64,f64
"""total_rows""",8029895,100.0
"""duplicate_rows""",0,0.0
"""null_pickup_datetime""",0,0.0
"""null_dropoff_datetime""",0,0.0
"""non_positive_trip_distance""",278394,3.467
"""non_positive_trip_duration""",103091,1.2838
"""negative_fare_amount""",87638,1.0914
"""negative_total_amount""",88853,1.1065
"""zero_distance_positive_fare""",271010,3.375


In [33]:
daily_trip_counts = (
    eda_df.group_by("pickup_date")
    .agg(pl.len().alias("trip_count"))
    .sort("pickup_date")
)

alt.Chart(daily_trip_counts).mark_line(point=True).encode(
    x=alt.X("pickup_date:T", title="Pickup date"),
    y=alt.Y("trip_count:Q", title="Trip count"),
    tooltip=["pickup_date", "trip_count"],
).properties(
    width=700,
    height=350,
    title="Daily Trip Volume"
)

alt.Chart(...)

In [34]:
weekday_labels = pl.DataFrame(
    {
        "pickup_weekday": [1, 2, 3, 4, 5, 6, 7],
        "weekday_name": [
            "Mon",
            "Tue",
            "Wed",
            "Thu",
            "Fri",
            "Sat",
            "Sun",
        ],
    }
)

weekday_summary = (
    eda_df.group_by("pickup_weekday")
    .agg(
        pl.len().alias("trip_count"),
        pl.mean("trip_duration_minutes").round(2).alias("avg_duration_min"),
        pl.mean("trip_distance").round(2).alias("avg_distance"),
        pl.mean("total_amount").round(2).alias("avg_total_amount"),
    )
    .sort("pickup_weekday")
    .join(weekday_labels, on="pickup_weekday", how="left")
)

alt.Chart(weekday_summary).mark_bar().encode(
    x=alt.X("weekday_name:O", title="Weekday", sort=alt.EncodingSortField(field="pickup_weekday", order="ascending")),
    y=alt.Y("trip_count:Q", title="Trip count"),
    tooltip=["weekday_name", "trip_count", "avg_duration_min", "avg_distance", "avg_total_amount"],
).properties(
    width=700,
    height=350,
    title="Trip Count by Weekday"
)

alt.Chart(...)

In [35]:
fare_distance_sample = eda_df.select(
    ["trip_distance", "fare_amount", "total_amount", "trip_duration_minutes", "pickup_hour"]
).sample(
    n=min(10000, eda_df.height),
    seed=42,
)

alt.Chart(fare_distance_sample).mark_circle(size=28, opacity=0.35).encode(
    x=alt.X("trip_distance:Q", title="Trip distance"),
    y=alt.Y("fare_amount:Q", title="Fare amount"),
    color=alt.Color("pickup_hour:Q", title="Pickup hour"),
    tooltip=["trip_distance", "fare_amount", "total_amount", "trip_duration_minutes", "pickup_hour"],
).properties(
    width=700,
    height=400,
    title="Trip Distance vs Fare Amount"
)

MaxRowsError: The number of rows in your dataset (10000) is greater than the maximum allowed (5000).

Try enabling the VegaFusion data transformer which raises this limit by pre-evaluating data
transformations in Python.
    >> import altair as alt
    >> alt.data_transformers.enable("vegafusion")

Or, see https://altair-viz.github.io/user_guide/large_datasets.html for additional information
on how to plot large datasets.

alt.Chart(...)

In [36]:
feature_summary = eda_df.select(
    [
        pl.len().alias("row_count"),
        pl.mean("trip_duration_minutes").round(2).alias("avg_trip_duration_min"),
        pl.mean("trip_distance").round(2).alias("avg_trip_distance"),
        pl.mean("fare_amount").round(2).alias("avg_fare_amount"),
        pl.mean("total_amount").round(2).alias("avg_total_amount"),
        pl.median("passenger_count").alias("median_passenger_count"),
    ]
)

candidate_features = [
    "pickup_hour",
    "pickup_weekday",
    "is_weekend",
    "trip_distance",
    "passenger_count",
    "PULocationID",
    "DOLocationID",
]

target_column = "trip_duration_minutes"

print("Candidate feature columns for modeling:")
for feature in candidate_features:
    print(f" - {feature}")

print(f"\nSuggested target column: {target_column}")
display(feature_summary)

Candidate feature columns for modeling:
 - pickup_hour
 - pickup_weekday
 - is_weekend
 - trip_distance
 - passenger_count
 - PULocationID
 - DOLocationID

Suggested target column: trip_duration_minutes


row_count,avg_trip_duration_min,avg_trip_distance,avg_fare_amount,avg_total_amount,median_passenger_count
u32,f64,f64,f64,f64,f64
7564562,18.07,3.5,21.8,30.55,1.0


## Exact Target Definition

The prediction target for this project is `trip_duration_minutes`.

It is defined as:

$$
\text{trip\_duration\_minutes} = \frac{\text{tpep\_dropoff\_datetime} - \text{tpep\_pickup\_datetime}}{60\ \text{seconds}}
$$

This target is derived from raw trip timestamps and should be treated as the label to predict, **not** as an input feature.

For a pickup-time prediction task, `tpep_dropoff_datetime` is a leakage column and must not be used as a model feature.

In [37]:
invalid_rule_summary = pl.DataFrame(
    {
        "rule": [
            "null pickup timestamp",
            "null dropoff timestamp",
            "non-positive trip duration",
            "trip duration over 180 minutes",
            "non-positive trip distance",
            "trip distance over 50 miles",
            "negative fare amount",
            "negative total amount",
        ],
        "affected_rows": [
            all_data_df.filter(pl.col("tpep_pickup_datetime").is_null()).height,
            all_data_df.filter(pl.col("tpep_dropoff_datetime").is_null()).height,
            all_data_df.filter(pl.col("trip_duration_minutes") <= 0).height,
            all_data_df.filter(pl.col("trip_duration_minutes") > 180).height,
            all_data_df.filter(pl.col("trip_distance") <= 0).height,
            all_data_df.filter(pl.col("trip_distance") > 50).height,
            all_data_df.filter(pl.col("fare_amount") < 0).height,
            all_data_df.filter(pl.col("total_amount") < 0).height,
        ],
    }
).with_columns(
    (pl.col("affected_rows") / all_data_df.height * 100).round(4).alias("pct_of_rows")
)

recommended_invalid_row_rules = [
    "drop rows with null tpep_pickup_datetime",
    "drop rows with null tpep_dropoff_datetime",
    "drop rows with trip_duration_minutes <= 0",
    "drop rows with trip_duration_minutes > 180 for the first-pass model dataset",
    "drop rows with trip_distance <= 0",
    "drop rows with trip_distance > 50 for the first-pass model dataset",
    "drop rows with fare_amount < 0",
    "drop rows with total_amount < 0",
]

display(invalid_rule_summary)

print("Recommended first-pass invalid row rules:")
for rule in recommended_invalid_row_rules:
    print(f" - {rule}")

rule,affected_rows,pct_of_rows
str,i64,f64
"""null pickup timestamp""",0,0.0
"""null dropoff timestamp""",0,0.0
"""non-positive trip duration""",103091,1.2838
"""trip duration over 180 minutes""",3915,0.0488
"""non-positive trip distance""",278394,3.467
"""trip distance over 50 miles""",1358,0.0169
"""negative fare amount""",87638,1.0914
"""negative total amount""",88853,1.1065


Recommended first-pass invalid row rules:
 - drop rows with null tpep_pickup_datetime
 - drop rows with null tpep_dropoff_datetime
 - drop rows with trip_duration_minutes <= 0
 - drop rows with trip_duration_minutes > 180 for the first-pass model dataset
 - drop rows with trip_distance <= 0
 - drop rows with trip_distance > 50 for the first-pass model dataset
 - drop rows with fare_amount < 0
 - drop rows with total_amount < 0


## First Modeling Feature Set

The first-pass modeling feature set should emphasize variables that are available at prediction time.

### Approved first-pass features

- `pickup_hour`
- `pickup_weekday`
- `is_weekend`
- `trip_distance`
- `passenger_count`
- `PULocationID`

### Conditional feature

- `DOLocationID` can be included only if destination is known at prediction time.

### Leakage exclusions

The following fields should not be used as model inputs for a pickup-time prediction task:

- `tpep_dropoff_datetime`
- `trip_duration_minutes`
- other post-trip outcome fields derived after ride completion

In [38]:
split_range_summary = all_data_df.select(
    [
        pl.min("tpep_pickup_datetime").alias("min_pickup_datetime"),
        pl.max("tpep_pickup_datetime").alias("max_pickup_datetime"),
    ]
)

monthly_trip_counts = (
    all_data_df
    .with_columns(pl.col("tpep_pickup_datetime").dt.strftime("%Y-%m").alias("pickup_month"))
    .group_by("pickup_month")
    .agg(pl.len().alias("trip_count"))
    .sort("pickup_month")
)

display(split_range_summary)
display(monthly_trip_counts)

print("Recommended time-based split strategy:")
print(" - Sort data by tpep_pickup_datetime")
print(" - Train on earlier trips")
print(" - Validate on the next chronological slice")
print(" - Test on the most recent held-out slice")
print(" - Do not use a random split")

if monthly_trip_counts.height >= 2:
    months = monthly_trip_counts.get_column("pickup_month").to_list()
    print("\nCurrent chronological month coverage:")
    for month in months:
        print(f" - {month}")
    print("\nPractical first-pass proposal:")
    print(f" - Train on: {', '.join(months[:-1])}")
    print(f" - Test on: {months[-1]}")
    print(" - If more data is added later, carve out a validation window immediately before the test window")
else:
    print("\nOnly one month is currently available, so use an early/late chronological split within that month for validation and testing.")

min_pickup_datetime,max_pickup_datetime
datetime[μs],datetime[μs]
2025-11-30 20:48:56,2026-02-01 00:45:01


pickup_month,trip_count
str,u32
"""2025-11""",9
"""2025-12""",4305003
"""2026-01""",3724882
"""2026-02""",1


Recommended time-based split strategy:
 - Sort data by tpep_pickup_datetime
 - Train on earlier trips
 - Validate on the next chronological slice
 - Test on the most recent held-out slice
 - Do not use a random split

Current chronological month coverage:
 - 2025-11
 - 2025-12
 - 2026-01
 - 2026-02

Practical first-pass proposal:
 - Train on: 2025-11, 2025-12, 2026-01
 - Test on: 2026-02
 - If more data is added later, carve out a validation window immediately before the test window


## Proposed Time-Based Split Strategy

This project should use a **chronological** split based on `tpep_pickup_datetime`.

### Decision

- sort observations by pickup timestamp
- train on earlier trips
- validate on later trips
- test on the most recent held-out slice
- do **not** use a random split

### Reason

Trip behavior changes across dates, weekdays, and hours. A chronological split better simulates production batch prediction and avoids leaking future behavior into training.

### Practical first-pass policy

- if multiple months are available, train on earlier month(s) and test on the latest month
- if only one month is available, use an early/late split within that month
- when more history is added, reserve a validation period immediately before the final test period

## EDA Conclusion and Recommended Next Step
This notebook is now focused on **exploration and validation** of the raw NYC taxi dataset.

### Confirmed Phase 1 decisions

#### Exact target definition
- target column: `trip_duration_minutes`
- definition: `tpep_dropoff_datetime - tpep_pickup_datetime`, expressed in minutes
- target is derived from raw timestamps and is the prediction label, not a feature

#### First-pass invalid row rules
- require non-null `tpep_pickup_datetime`
- require non-null `tpep_dropoff_datetime`
- require `trip_duration_minutes > 0`
- cap `trip_duration_minutes` at 180 minutes for the first-pass model dataset
- require `trip_distance > 0`
- cap `trip_distance` at 50 for the first-pass model dataset
- require `fare_amount >= 0`
- require `total_amount >= 0`

#### First modeling feature set
- `pickup_hour`
- `pickup_weekday`
- `is_weekend`
- `trip_distance`
- `passenger_count`
- `PULocationID`
- `DOLocationID` only if destination is known at prediction time

#### Leakage exclusions
- `tpep_dropoff_datetime`
- `trip_duration_minutes` as an input feature
- any post-trip outcome field only known after trip completion

#### Time-based split decision
- use a chronological split based on `tpep_pickup_datetime`
- train on earlier trips
- validate on the next chronological slice
- test on the most recent held-out slice
- do not use a random split

### Recommended next artifact
The next step is to formalize these decisions into transformation logic:
- create the initial `dbt_taxi/` project
- write a silver model for cleaning rules
- write a gold model for the first feature set

This keeps `01_data_exploration.ipynb` as the evidence base for the first production-style data rules.